In [2]:
import numpy as np, pandas as pd, string
from string import digits
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

lines = pd.read_csv("Hindi_English_Truncated_Corpus.csv", encoding='utf-8')
lines = lines[lines['source'] == 'ted'][['english_sentence', 'hindi_sentence']].dropna().drop_duplicates()
lines = lines.sample(n=25000, random_state=42)

In [3]:
lines

,english_sentence,hindi_sentence
82040,"We still don't know who her parents are, who s...",हम अभी तक नहीं जानते हैं कि उसके माता-पिता कौन...
85038,"no keyboard,","कोई कुंजीपटल नहीं,"
58018,"But as far as being a performer,",लेकिन एक कलाकार होने के साथ
74470,"And this particular balloon,","और यह खास गुब्बारा,"
122330,and it's not as hard as you think. Integrate c...,"और जितना आपको लगता है, यह उतना कठिन नहीं है.अप..."
...,...,...
49566,using either image recognition or marker techn...,छवि मान्यता या मार्कर प्रौद्योगिकी का इस्तेमाल...
118399,And they've started doing DNA tests on kids,और उन्होंने बच्चो पर DNA परीक्षण शुरू कर दिये है
20473,So there is not a lot of competition.,तो ज्यादा प्रतियोगिता नहीं है.
20729,"a woman with indefatigable stamina,",एक अजेय बलवाली महिला


In [4]:
def clean_text(text):
    exclude = set(string.punctuation)
    text = ''.join(ch for ch in text if ch not in exclude)
    text = text.translate(str.maketrans('', '', digits))
    return text.strip().lower()

In [5]:
lines['english_sentence'] = lines['english_sentence'].apply(clean_text)
lines['hindi_sentence'] = lines['hindi_sentence'].apply(clean_text)
lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x: 'start_ ' + x + ' _end')

In [6]:
lines['english_sentence']

82040     we still dont know who her parents are who she is
85038                                           no keyboard
58018                       but as far as being a performer
74470                           and this particular balloon
122330    and its not as hard as you think integrate cli...
                                ...                        
49566     using either image recognition or marker techn...
118399           and theyve started doing dna tests on kids
20473                  so there is not a lot of competition
20729                    a woman with indefatigable stamina
91889                  and you say “how about eight oclock”
Name: english_sentence, Length: 25000, dtype: str

In [7]:
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(lines['english_sentence'])
eng_seq = eng_tokenizer.texts_to_sequences(lines['english_sentence'])

hin_tokenizer = Tokenizer(filters='')  
hin_tokenizer.fit_on_texts(lines['hindi_sentence'])
hin_seq = hin_tokenizer.texts_to_sequences(lines['hindi_sentence'])

In [8]:
hin_tokenizer

In [9]:
hin_seq

[[1, 16, 166, 56, 15, 154, 7, 11, 145, 1274, 384, 7, 62, 384, 3, 2],
 [1, 61, 8166, 15, 2],
 [1, 59, 10, 878, 122, 5, 42, 2],
 [1, 4, 13, 776, 2704, 2],
 [1,
  4,
  575,
  65,
  104,
  3,
  13,
  804,
  457,
  15,
  8167,
  112,
  5648,
  6,
  2165,
  674,
  12,
  5649,
  209,
  2],
 [1, 4, 8168, 8169, 172, 1057, 210, 110, 2],
 [1, 13, 467, 200, 53, 8, 640, 8, 468, 3, 2],
 [1, 91, 24, 134, 12, 1162, 173, 14, 335, 15, 47, 31, 111, 659, 589, 3, 2],
 [1, 273, 38, 172, 844, 194, 344, 5, 4369, 5, 8170, 9, 32, 1721, 1603, 2],
 [1,
  2705,
  104,
  3,
  91,
  22,
  4370,
  1013,
  76,
  1058,
  56,
  469,
  128,
  3,
  41,
  22,
  4371,
  214,
  128,
  3,
  2],
 [1, 77, 2706, 139, 23, 2],
 [1, 126, 38, 879, 193, 7, 8171, 5650, 4, 38, 1108, 3, 40, 2],
 [1, 19, 17, 302, 527, 9, 1275, 5651, 5652, 99, 3, 2],
 [1, 281, 8, 8172, 4372, 64, 2],
 [1, 19, 2166, 87, 8, 41, 13, 38, 660, 1276, 6, 605, 48, 2],
 [1, 21, 17, 1722, 5653, 299, 109, 2401, 69, 17, 6, 845, 117, 30, 2],
 [1, 22, 3637, 107, 3, 67, 

In [10]:
max_eng_len = max(len(seq) for seq in eng_seq)
max_hin_len = max(len(seq) for seq in hin_seq)

encoder_input = pad_sequences(eng_seq, maxlen=max_eng_len, padding='post')
decoder_input = pad_sequences(hin_seq, maxlen=max_hin_len, padding='post')

In [11]:
encoder_input

array([[  11,  203,   68, ...,    0,    0,    0],
       [  84, 4936,    0, ...,    0,    0,    0],
       [  20,   30,  297, ...,    0,    0,    0],
       ...,
       [  14,   37,    9, ...,    0,    0,    0],
       [   5,  271,   23, ...,    0,    0,    0],
       [   2,   10,  106, ...,    0,    0,    0]],
      shape=(25000, 20), dtype=int32)

In [12]:
decoder_input

array([[   1,   16,  166, ...,    0,    0,    0],
       [   1,   61, 8166, ...,    0,    0,    0],
       [   1,   59,   10, ...,    0,    0,    0],
       ...,
       [   1,   19,  113, ...,    0,    0,    0],
       [   1,   10, 8161, ...,    0,    0,    0],
       [   1,    4, 3769, ...,    0,    0,    0]],
      shape=(25000, 30), dtype=int32)

In [13]:
decoder_target = np.zeros((decoder_input.shape[0], decoder_input.shape[1], 1))
decoder_target[:, 0:-1, 0] = decoder_input[:, 1:]

In [14]:
decoder_target

array([[[1.6000e+01],
        [1.6600e+02],
        [5.6000e+01],
        ...,
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00]],

       [[6.1000e+01],
        [8.1660e+03],
        [1.5000e+01],
        ...,
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00]],

       [[5.9000e+01],
        [1.0000e+01],
        [8.7800e+02],
        ...,
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00]],

       ...,

       [[1.9000e+01],
        [1.1300e+02],
        [1.4170e+03],
        ...,
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00]],

       [[1.0000e+01],
        [8.1610e+03],
        [1.7723e+04],
        ...,
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00]],

       [[4.0000e+00],
        [3.7690e+03],
        [7.6700e+02],
        ...,
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00]]], shape=(25000, 30, 1))

In [16]:
eng_vocab_size = len(eng_tokenizer.word_index) + 1
hin_vocab_size = len(hin_tokenizer.word_index) + 1
latent_dim = 256

encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(eng_vocab_size, latent_dim)(encoder_inputs)
enc_outputs, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

E0000 00:00:1778481542.666164   61193 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [17]:
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(hin_vocab_size, latent_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(hin_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [18]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')
model.fit([encoder_input, decoder_input], decoder_target, batch_size=64, epochs=20, validation_split=0.2)

Epoch 1/20


W0000 00:00:1778481668.302931   66154 cpu_allocator_impl.cc:82] Allocation of 136120320 exceeds 10% of free system memory.
W0000 00:00:1778481668.779276   66154 cpu_allocator_impl.cc:82] Allocation of 1161560064 exceeds 10% of free system memory.


  1/313 ━━━━━━━━━━━━━━━━━━━━ 24:06 5s/step - loss: 9.7854

W0000 00:00:1778481670.314873   66157 cpu_allocator_impl.cc:82] Allocation of 136120320 exceeds 10% of free system memory.
W0000 00:00:1778481671.057663   66151 cpu_allocator_impl.cc:82] Allocation of 1161560064 exceeds 10% of free system memory.


  2/313 ━━━━━━━━━━━━━━━━━━━━ 11:54 2s/step - loss: 9.7776

W0000 00:00:1778481672.622356   66153 cpu_allocator_impl.cc:82] Allocation of 136120320 exceeds 10% of free system memory.


 53/313 ━━━━━━━━━━━━━━━━━━━━ 8:38 2s/step - loss: 5.8523

KeyboardInterrupt: 